In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **CNN PROJECT**

**Imports and Dataset Loading**

In [ ]:
# Cell 1: Imports and dataset loading

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision

from torchvision.datasets import OxfordIIITPet

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

# Downloads into ./data/oxford-iiit-pet.
# Use split="trainval" for the provided training/validation portion.
dataset = OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="segmentation",
    download=True,
)

print(f"Number of samples: {len(dataset)}")

**Display several image/mask pairs and inspect their values**

In [ ]:
# Cell 2: Display several image/mask pairs and inspect their values

sample_indices = [0, 100, 500, 1000]

fig, axes = plt.subplots(
    nrows=len(sample_indices),
    ncols=2,
    figsize=(10, 4 * len(sample_indices)),
)

for row, index in enumerate(sample_indices):
    # Both objects are PIL images because no transforms were supplied.
    image, mask = dataset[index]

    # Convert to NumPy only for inspection and plotting.
    image_array = np.array(image)
    mask_array = np.array(mask)

    print(f"Sample {index}")
    print(f"  PIL image size (width, height): {image.size}")
    print(f"  PIL mask size  (width, height): {mask.size}")
    print(f"  Image array shape (H, W, C):   {image_array.shape}")
    print(f"  Mask array shape  (H, W):      {mask_array.shape}")
    print(f"  Unique mask values:            {np.unique(mask_array)}")
    print()

    axes[row, 0].imshow(image_array)
    axes[row, 0].set_title(f"Image — sample {index}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(
        mask_array,
        cmap="viridis",
        vmin=1,
        vmax=3,
        interpolation="nearest",
    )
    axes[row, 1].set_title(
        f"Trimap — values {np.unique(mask_array).tolist()}"
    )
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

**Preprocessing Dataset**

In [ ]:
# Cell 3: Preprocessing Dataset

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


class BinaryPetDataset(Dataset):
    """Wrap the existing Oxford-IIIT Pet dataset for binary segmentation."""

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, mask = self.base_dataset[index]

        # Ensure every input has three RGB channels.
        image = image.convert("RGB")

        # Resize images with bilinear interpolation.
        image = TF.resize(
            image,
            size=[128, 128],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        )

        # Resize masks with nearest-neighbor interpolation to preserve labels.
        mask = TF.resize(
            mask,
            size=[128, 128],
            interpolation=InterpolationMode.NEAREST,
        )

        # Converts the image to float32 [3, 128, 128] in the range [0, 1].
        image_tensor = TF.to_tensor(image)

        # Convert the mask to a long tensor [128, 128].
        mask_array = np.array(mask, dtype=np.int64, copy=True)
        mask_tensor = torch.from_numpy(mask_array)

        # Binary conversion:
        # 1 (pet) and 3 (border) -> 1
        # 2 (background)         -> 0
        mask_tensor = (
            (mask_tensor == 1) | (mask_tensor == 3)
        ).long()

        assert image_tensor.shape == (3, 128, 128)
        assert mask_tensor.shape == (128, 128)

        return image_tensor, mask_tensor


binary_dataset = BinaryPetDataset(dataset)

**DataLoader**

In [ ]:
# Cell 4: DataLoader

generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    binary_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    generator=generator,
)

images, masks = next(iter(train_loader))

print("Images:", images.shape)
print("Masks: ", masks.shape)
print("Unique mask values:", torch.unique(masks).tolist())
print("Image value range:", images.min().item(), "to", images.max().item())

**Visualize one preprocessed sample**

In [ ]:
# Cell 5: Visualize one preprocessed sample

image = images[0].permute(1, 2, 0).cpu().numpy()
mask = masks[0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(image)
axes[0].set_title("Preprocessed image")
axes[0].axis("off")

axes[1].imshow(
    mask,
    cmap="gray",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[1].set_title("Binary mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

**Tiny U-Net architecture**

* What is this idea of having an putting parenthesis after a attribute
* Decoder understanding (up, cat, decoder)
* Use all decoders?

In [ ]:
# Cell 6: Tiny U-Net architecture

import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    """Two convolution layers, each followed by BatchNorm and ReLU."""

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TinyUNet(nn.Module):
    """
    Small U-Net for two-class semantic segmentation.

    Input:  [B, 3, H, W]
    Output: [B, 2, H, W] raw logits
    """

    def __init__(
        self,
        in_channels=3,
        num_classes=2,
        base_channels=32,
        bottleneck_dropout=0.2,
    ):
        super().__init__()

        c = base_channels

        # Encoder
        self.encoder1 = ConvBlock(in_channels, c)
        self.encoder2 = ConvBlock(c, c * 2)
        self.encoder3 = ConvBlock(c * 2, c * 4)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = ConvBlock(c * 4, c * 8)
        self.dropout = nn.Dropout2d(bottleneck_dropout)

        # Decoder
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.decoder3 = ConvBlock(c * 8, c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.decoder2 = ConvBlock(c * 4, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.decoder1 = ConvBlock(c * 2, c)

        # One raw logit per class at every pixel.
        self.output_layer = nn.Conv2d(c, num_classes, kernel_size=1)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError(f"Expected [B, 3, H, W], received {x.shape}")

        # Three pooling operations require dimensions divisible by 8.
        if x.shape[-2] % 8 != 0 or x.shape[-1] % 8 != 0:
            raise ValueError("Image height and width must be divisible by 8.")

        # Encoder
        e1 = self.encoder1(x)          # [B, 32, 128, 128]
        e2 = self.encoder2(self.pool(e1))  # [B, 64, 64, 64]
        e3 = self.encoder3(self.pool(e2))  # [B, 128, 32, 32]

        # Bottleneck
        b = self.bottleneck(self.pool(e3))  # [B, 256, 16, 16]
        b = self.dropout(b)

        # Decoder with skip connection from encoder3.
        d3 = self.up3(b)                    # [B, 128, 32, 32]
        d3 = torch.cat([d3, e3], dim=1)    # [B, 256, 32, 32]
        d3 = self.decoder3(d3)              # [B, 128, 32, 32]

        # Skip connection from encoder2.
        d2 = self.up2(d3)                   # [B, 64, 64, 64]
        d2 = torch.cat([d2, e2], dim=1)    # [B, 128, 64, 64]
        d2 = self.decoder2(d2)              # [B, 64, 64, 64]

        # Skip connection from encoder1.
        d1 = self.up1(d2)                   # [B, 32, 128, 128]
        d1 = torch.cat([d1, e1], dim=1)    # [B, 64, 128, 128]
        d1 = self.decoder1(d1)              # [B, 32, 128, 128]

        # No sigmoid or softmax: CrossEntropyLoss expects raw logits.
        logits = self.output_layer(d1)      # [B, 2, 128, 128]

        return logits

**Device setup and forward-pass smoke test**

* next and iter need help
* Need more review with .eval and .inference_mode
* what is requires_grad

In [ ]:
# Cell 7: Device setup and forward-pass smoke test

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_channels=32,
    bottleneck_dropout=0.2,
).to(device)

images, masks = next(iter(train_loader))
images = images.to(device)
masks = masks.to(device)

# Evaluation mode avoids updating BatchNorm statistics during the smoke test.
model.eval()

with torch.inference_mode():
    outputs = model(images)

print("Device: ", device)
print("Images: ", images.shape)
print("Outputs:", outputs.shape)
print("Masks:  ", masks.shape)

expected_shape = (images.shape[0], 2, 128, 128)

assert outputs.shape == expected_shape, (
    f"Expected output shape {expected_shape}, received {outputs.shape}"
)
assert masks.shape == (images.shape[0], 128, 128)
assert masks.dtype == torch.long
assert torch.isfinite(outputs).all(), "Model produced NaN or infinite logits."

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Trainable parameters: {trainable_parameters:,}")
print("Forward-pass smoke test passed.")

**Set up the single-batch overfit test**

In [ ]:
# Cell 8: Set up the single-batch overfit test

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a fresh model specifically for this debugging test.
# Dropout is disabled to make memorization more deterministic.
overfit_model = TinyUNet(
    in_channels=3,
    num_classes=2,
    base_channels=32,
    bottleneck_dropout=0.0,
).to(device)

# CrossEntropyLoss expects:
# logits: [B, 2, H, W]
# labels: [B, H, W] with dtype torch.long
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    overfit_model.parameters(),
    lr=1e-3,
)

# Grab one batch once. This exact batch is reused for all 100 steps.
images, masks = next(iter(train_loader))
images = images.to(device)
masks = masks.to(device)

assert images.shape[1:] == (3, 128, 128)
assert masks.shape[1:] == (128, 128)
assert masks.dtype == torch.long

print("Device:", device)
print("Fixed image batch:", images.shape)
print("Fixed mask batch: ", masks.shape)

**Train repeatedly on the same fixed batch**

In [ ]:
# Cell 9: Train repeatedly on the same fixed batch

overfit_model.train()
losses = []

for step in range(1, 101):
    # Clear gradients left over from the previous step.
    optimizer.zero_grad()

    # Raw logits—do not apply softmax before CrossEntropyLoss.
    outputs = overfit_model(images)

    # Compare each pixel's two class logits with its integer class label.
    loss = criterion(outputs, masks)

    # Compute gradients and update the model weights.
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if step % 10 == 0:
        print(f"Step {step:3d}/100 | Loss: {loss.item():.6f}")

starting_loss = losses[0]
final_loss = losses[-1]
percent_decrease = (
    (starting_loss - final_loss) / starting_loss
) * 100

print(f"\nStarting loss:   {starting_loss:.6f}")
print(f"Final loss:      {final_loss:.6f}")
print(f"Percent decrease: {percent_decrease:.2f}%")

assert final_loss < starting_loss, (
    "The final loss was not lower than the starting loss."
)

**Plot the optimization curve**

In [ ]:
# Cell 10: Plot the optimization curve

plt.figure(figsize=(8, 4))
plt.plot(range(1, 101), losses)
plt.xlabel("Optimization step")
plt.ylabel("Cross-entropy loss")
plt.title("Single-Batch Overfit Test")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Evaluate the memorized batch**

In [ ]:
# Cell 11: Evaluate the memorized batch

overfit_model.eval()

with torch.inference_mode():
    outputs = overfit_model(images)
    predictions = outputs.argmax(dim=1)

# Pixel accuracy is useful for debugging, but can be inflated by background.
pixel_accuracy = (predictions == masks).float().mean().item()

# Foreground IoU gives a more informative segmentation check.
intersection = ((predictions == 1) & (masks == 1)).sum()
union = ((predictions == 1) | (masks == 1)).sum()
foreground_iou = (
    intersection.float() / union.clamp_min(1)
).item()

print(f"Fixed-batch pixel accuracy: {pixel_accuracy:.2%}")
print(f"Fixed-batch foreground IoU: {foreground_iou:.2%}")

**Interpretation/Reflection**

In [ ]:
# Cell 13: Interpretation

print("""
Single-batch overfit test:

If the loss drops substantially:
- The forward pass, loss calculation, backpropagation, optimizer, and
  parameter updates are probably connected correctly.
- The model has enough capacity to learn this small fixed batch.

This does not prove:
- That the model will generalize to unseen images.
- That the full dataset is correctly split or balanced.
- That the final training settings will produce a good model.

If the loss does not drop, check:
- Logit and mask shapes.
- That masks are torch.long and contain only 0 and 1.
- That no sigmoid or softmax is used before CrossEntropyLoss.
- That training is not inside torch.inference_mode().
- That model.train(), loss.backward(), and optimizer.step() are called.
- That the optimizer received the fresh model's parameters.
- That the same fixed batch is reused on every step.
""")